# 02 — ICD-10 Mapping

Demonstrate the three-step ICD-10 mapping pipeline.

**What this notebook covers**
- Exact match
- Fuzzy match (rapidfuzz)
- Embedding match (sentence-transformers)
- End-to-end: NER → ICD-10
- Mapping quality analysis

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from src.nlp.icd_mapper import ICD10Mapper
from src.utils.config import Paths

print('Setup OK')

Setup OK


## 2. Load the ICD-10 Mapper

Run the ETL pipeline first to generate `data/processed/icd10_codes.parquet`:
```bash
python -m src.etl.pipeline --dry-run
```

In [2]:
mapper = ICD10Mapper()
print(f'Codes loaded: {len(mapper._df):,}')

2026-06-26 19:07:46 | INFO     | src.nlp.icd_mapper             | ICD10Mapper ready: 74260 codes loaded
Codes loaded: 74,260


## 3. Exact Match

In [3]:
test_terms = [
    'essential hypertension',
    'type 2 diabetes mellitus',
    'unspecified pneumonia',
]

for term in test_terms:
    matches = mapper.map(term)
    if matches:
        m = matches[0]
        print(f'{term:<40}  {m.icd10_code}  [{m.match_method}]  {m.description}')
    else:
        print(f'{term:<40}  no match')

essential hypertension                    I10  [exact]  Essential (primary) hypertension
type 2 diabetes mellitus                  E11.9  [fuzzy]  Type 2 diabetes mellitus without complications
unspecified pneumonia                     J18.9  [fuzzy]  Pneumonia, unspecified organism


## 4. Fuzzy Match — Handles Spelling Variants

In [4]:
fuzzy_terms = [
    'hypertensioon',         # typo
    'diabetees type 2',      # typo
    'pneumonnia',            # typo
    'heart failure congest', # abbreviated
]

for term in fuzzy_terms:
    matches = mapper.map(term)
    if matches:
        m = matches[0]
        print(f'{term:<35}  {m.icd10_code}  [{m.match_method} {m.confidence:.0%}]  {m.description}')
    else:
        print(f'{term:<35}  no match')

2026-06-26 19:09:45 | INFO     | sentence_transformers.base.model | No device provided, using cpu
2026-06-26 19:09:48 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-06-26 19:09:48 | INFO     | sentence_transformers.base.model | Loading SentenceTransformer model from pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb.
2026-06-26 19:09:49 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/models/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-06-26 19:09:50 | INFO     | src.nlp.icd_mapper             | Loaded embedding model from local cache (offline): pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
2026-06-26 19:09:50 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/models/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb/commits/main "HTTP/1.1 200 OK"
2026-06-26 19:09:51 | INFO     | src.nlp.icd_mapper             | Loaded cached ICD-10 embedding index from C:\Users\Hp\Documents\clinical-nlp-pipeline\data\processed\icd10_embeddings.pt (74260 vectors)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-26 19:09:51 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/models/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb/discussions?p=0 "HTTP/1.1 200 OK"
2026-06-26 19:09:52 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/models/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb/commits/refs%2Fpr%2F5 "HTTP/1.1 200 OK"
2026-06-26 19:09:52 | INFO     | httpx                          | HTTP Request: HEAD https://huggingface.co/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb/resolve/refs%2Fpr%2F5/model.safetensors.index.json "HTTP/1.1 404 Not Found"
hypertensioon                        I10  [embedding 74%]  Essential (primary) hypertension


...

diabetees type 2                     E11.65  [embedding 73%]  Type 2 diabetes mellitus with hyperglycemia


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

pneumonnia                           J84.114  [embedding 77%]  Acute interstitial pneumonitis


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

heart failure congest                I50.22  [embedding 83%]  Chronic systolic (congestive) heart failure


## 5. Embedding Match — Handles Novel Phrasing

In [5]:
# These will fall through exact and fuzzy, reaching the embedding matcher
# Note: first call builds the embedding index (~30 seconds)
novel_terms = [
    'difficulty breathing on exertion',
    'elevated blood glucose in non-insulin dependent patient',
    'lung infection with fever and cough',
]

print('Building embedding index (first call only)...')
for term in novel_terms:
    matches = mapper.map(term)
    if matches:
        m = matches[0]
        print(f'\n  Query   : {term}')
        print(f'  Match   : {m.icd10_code}  {m.description}')
        print(f'  Method  : {m.match_method}  confidence {m.confidence:.2f}')
    else:
        print(f'\n  Query   : {term}')
        print(f'  Match   : none above threshold')

Building embedding index (first call only)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Query   : difficulty breathing on exertion
  Match   : R07.1  Chest pain on breathing
  Method  : embedding  confidence 0.71


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Query   : elevated blood glucose in non-insulin dependent patient
  Match   : E72.51  Non-ketotic hyperglycinemia
  Method  : embedding  confidence 0.72


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Query   : lung infection with fever and cough
  Match   : R05.1  Acute cough
  Method  : embedding  confidence 0.71


## 6. End-to-End: NER → ICD-10

In [6]:
from src.nlp.ner import build_ner_pipeline
from src.utils.text_utils import clean_clinical_text

ner  = build_ner_pipeline()
note = """
Patient admitted with acute MI. History of hypertension and
type 2 diabetes. Started on aspirin, metoprolol, and lisinopril.
ECG shows ST elevation. Urgent cardiac catheterisation planned.
"""

cleaned  = clean_clinical_text(note)
entities = ner.extract(cleaned)
icd_map  = mapper.map_entities(entities)

print('Entity → ICD-10 mappings:')
for ent_text, matches in icd_map.items():
    best = matches[0]
    print(f'  {ent_text:<35}  {best.icd10_code}  {best.description}')

2026-06-26 19:10:08 | INFO     | src.nlp.ner                    | Loading NER model: en_ner_bc5cdr_md
2026-06-26 19:10:34 | INFO     | src.nlp.ner                    | NER model loaded ✓
2026-06-26 19:10:35 | INFO     | src.nlp.ner                    | Loading NER model: en_core_sci_lg
2026-06-26 19:11:00 | INFO     | src.nlp.ner                    | NER model loaded ✓


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Entity → ICD-10 mappings:
  acute myocardial infarction          I21.9  Acute myocardial infarction, unspecified
  hypertension                         I10  Essential (primary) hypertension
  diabetes                             E08.65  Diabetes mellitus due to underlying condition with hyperglycemia
